In [1]:
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv

In [2]:
load_dotenv()
model = ChatGroq(model="llama-3.1-8b-instant",temperature=1.5)

In [3]:
# step 1: define states
class BlogState(TypedDict):
    title: str
    outline:str
    content:str
    evaluation:str

In [4]:

def create_outline(state: BlogState) -> BlogState:
    title = state["title"]
    prompt =f"Generate a 5 lines of outline for a blog on the topic: {title}"
    outline = model.invoke(prompt).content

    state['outline'] = outline
    return state

In [5]:
def create_blog(state: BlogState) -> BlogState:
    title = state["title"]
    outline = state["outline"]
    prompt =f"write a 2 paragraph of  blog on the topic: {title} using the following outline \n {outline}"
    content = model.invoke(prompt).content

    state['content'] = content
    return state

In [6]:
def evaluate_blog(state: BlogState) -> BlogState:
    title = state["title"]
    outline = state["outline"]
    prompt =f"a statistical Evaluate the blog post with the topic: {title} using the following outline \n {outline}"
    evaluation = model.invoke(prompt).content

    state['evaluation'] = evaluation
    return state

In [7]:
# step 2: define graph
graph = StateGraph(BlogState)


# step 3: add nodes
graph.add_node("create_outline", create_outline) # require a title and res is outline
graph.add_node("create_blog", create_blog) # title and outline then res content
graph.add_node('evaluate_blog',evaluate_blog) # content and outline then res evaluation


# step 4: add edges
graph.add_edge(START, "create_outline")
graph.add_edge("create_outline", "create_blog")
graph.add_edge("create_blog", 'evaluate_blog')
graph.add_edge("evaluate_blog", END)

# step 5: compile
workflow = graph.compile()

In [12]:
inital_state = {'title': 'AI jobs in India'}

final_state = workflow.invoke(inital_state)

print(final_state['evaluation'])

**Statistical Evaluation of the Blog Post**

**Demographics:**

* Target audience: professionals, students, and educators interested in AI and its applications
* Location-based: Indian job seekers and those interested in AI careers in India

**Content Analysis:**

* **Section-wise Coverage:**
	+ Section I: Introduction (15% of total content)
		- Relevance score: 7/10 ( brief introduction and overview)
		- Engagement potential: 8/10 (intrigues readers about the growing AI industry)
	+ Section II: Emerging Job Roles (20% of total content)
		- Relevance score: 8/10 (covers various AI-related job roles)
		- Engagement potential: 8/10 (explains the skills and demand for AI-related roles)
	+ Section III: Talent Gap (20% of total content)
		- Relevance score: 8/10 (touches on the current talent shortage in the AI industry)
		- Engagement potential: 6/10 (somewhat brief discussion on educational institutions' initiations)
	+ Section IV: Sector-wise Opportunities (30% of total content)
		- Rele

In [13]:
print(final_state['outline'])

Here's a 5-line outline for a blog on AI jobs in India:

I. **Introduction**
  - Overview of the growing AI industry in India
  - Rising demand for AI professionals and associated job roles

II. **Emerging AI Job Roles**
  - Roles such as Machine Learning Engineers, AI Data Scientists, and NLP Specialists
  - Increasing demand for Cloud Computing and DevOps skills to support AI

III. **Talent Gap and Skilling Challenges**
  - Current shortage of talented professionals to bridge the gap between tech advancement and manpower supply
  - Educational institutions' initiatives for upskilling the workforce in AI and its fields

IV. **Job Opportunities Across Sectors**
  - Expanding use of AI in industries such as automotive, healthcare, finance, and e-commerce
  - Growing opportunities with startups and established companies in sectors that involve AI
